In [1]:
from pathlib import Path
import sys
import os
import random

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src").exists():
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root / "code"))

from import_libraries import *
import common_libraries.generic_library as gn_l
import common_libraries.io_library as io_l

from ml_classification.ml_classification_pipeline import (
    GeospatialClassificationPipeline,
    ClassificationPipelineConfig,
)

# Global reproducibility seed
random_seed = 42
os.environ["PYTHONHASHSEED"] = str(random_seed)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
random.seed(random_seed)
np.random.seed(random_seed)
try:
    import tensorflow as tf
    tf.random.set_seed(random_seed)
except Exception:
    print("TensorFlow not available, skipping TensorFlow seed setting.")

c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\xicor\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [2]:
project_dir = "C:/work_dir/kozani_parcels/ml_classification"
input_file = "C:/work_dir/kozani_parcels/final_parcel_statistics.parquet"

class_column = "label"
parcel_id_column = "parcel_code"

In [3]:
gdf = io_l.read_data(input_file)
gdf.head(3)

Reading file: C:/work_dir/kozani_parcels/final_parcel_statistics.parquet


,parcel_code,label,geometry,period_start,period_end,eligible_pixel_count,meets_minimum_pixel_count,B02_mean,B02_median,B02_sd,B02_min,B02_max,B02_p10,B02_p25,B02_p75,B02_p90,B03_mean,B03_median,B03_sd,B03_min,B03_max,B03_p10,B03_p25,B03_p75,B03_p90,B04_mean,B04_median,B04_sd,B04_min,B04_max,B04_p10,B04_p25,B04_p75,B04_p90,B05_mean,B05_median,B05_sd,B05_min,B05_max,B05_p10,B05_p25,B05_p75,B05_p90,B08_mean,B08_median,B08_sd,B08_min,B08_max,B08_p10,B08_p25,B08_p75,B08_p90,B11_mean,B11_median,B11_sd,B11_min,B11_max,B11_p10,B11_p25,B11_p75,B11_p90,B12_mean,B12_median,B12_sd,B12_min,B12_max,B12_p10,B12_p25,B12_p75,B12_p90,EVI_mean,EVI_median,EVI_sd,EVI_min,EVI_max,EVI_p10,EVI_p25,EVI_p75,EVI_p90,NDMI_mean,NDMI_median,NDMI_sd,NDMI_min,NDMI_max,NDMI_p10,NDMI_p25,NDMI_p75,NDMI_p90,NDRE_mean,NDRE_median,NDRE_sd,NDRE_min,NDRE_max,NDRE_p10,NDRE_p25,NDRE_p75,NDRE_p90,NDVI_mean,NDVI_median,NDVI_sd,NDVI_min,NDVI_max,NDVI_p10,NDVI_p25,NDVI_p75,NDVI_p90,NDWI_mean,NDWI_median,NDWI_sd,NDWI_min,NDWI_max,NDWI_p10,NDWI_p25,NDWI_p75,NDWI_p90,SAVI_mean,SAVI_median,SAVI_sd,SAVI_min,SAVI_max,SAVI_p10,SAVI_p25,SAVI_p75,SAVI_p90,batch_number,parcel_area_m2,approx_pixel_count,B02_range,B02_variance,B02_coefficient_of_variation,B02_iqr,B02_bowley_skewness,B02_p90_p10_spread,B03_range,B03_variance,B03_coefficient_of_variation,B03_iqr,B03_bowley_skewness,B03_p90_p10_spread,B04_range,B04_variance,B04_coefficient_of_variation,B04_iqr,B04_bowley_skewness,B04_p90_p10_spread,B05_range,B05_variance,B05_coefficient_of_variation,B05_iqr,B05_bowley_skewness,B05_p90_p10_spread,B08_range,B08_variance,B08_coefficient_of_variation,B08_iqr,B08_bowley_skewness,B08_p90_p10_spread,B11_range,B11_variance,B11_coefficient_of_variation,B11_iqr,B11_bowley_skewness,B11_p90_p10_spread,B12_range,B12_variance,B12_coefficient_of_variation,B12_iqr,B12_bowley_skewness,B12_p90_p10_spread,EVI_range,EVI_variance,EVI_coefficient_of_variation,EVI_iqr,EVI_bowley_skewness,EVI_p90_p10_spread,NDMI_range,NDMI_variance,NDMI_coefficient_of_variation,NDMI_iqr,NDMI_bowley_skewness,NDMI_p90_p10_spread,NDRE_range,NDRE_variance,NDRE_coefficient_of_variation,NDRE_iqr,NDRE_bowley_skewness,NDRE_p90_p10_spread,NDVI_range,NDVI_variance,NDVI_coefficient_of_variation,NDVI_iqr,NDVI_bowley_skewness,NDVI_p90_p10_spread,NDWI_range,NDWI_variance,NDWI_coefficient_of_variation,NDWI_iqr,NDWI_bowley_skewness,NDWI_p90_p10_spread,SAVI_range,SAVI_variance,SAVI_coefficient_of_variation,SAVI_iqr,SAVI_bowley_skewness,SAVI_p90_p10_spread
0,266124977,110,POLYGON ((346567...,2023-10-01,2023-11-01,44,True,0.0282545442,0.0284000002,0.0043494695,0.0165999997,0.0419999994,0.0238299984,0.0264749993,0.0296250004,0.0330500007,0.0596181804,0.0604499988,0.0069290514,0.0384999998,0.0725999996,0.0505199991,0.0571249984,0.0637249947,0.0658399984,0.0443272753,0.0423999988,0.0099631489,0.0339999981,0.0741999969,0.0354399979,0.0389249995,0.0441249982,0.0611600056,0.1105750041,0.1098999977,0.0042866542,0.1015999988,0.1198999956,0.1053499952,0.1080749929,0.1138499975,0.1161799952,0.3134113875,0.3127499819,0.0305520907,0.2403999865,0.3691999912,0.2760800123,0.2879999876,0.3371999860,0.3518100083,0.2462272644,0.2449999899,0.0125217816,0.2220999897,0.2701999843,0.2285500020,0.2406999916,0.2567249835,0.2635499835,0.1352249926,0.1349499971,0.0047329184,0.1275999993,0.1424999982,0.1291299909,0.1308249980,0.1398249865,0.1412099898,0.4948683219,0.4957464933,0.0646715824,0.3615827262,0.6074090004,0.3960260451,0.4575466812,0.5448441505,0.5721714497,0.1241611676,0.1165289432,0.0366340953,0.0431355536,0.2099553347,0.0839756578,0.0990398973,0.1452727318,0.1747567952,0.4807150147,0.4878600836,0.0401190688,0.3768955469,0.5515547991,0.4227118492,0.4497129023,0.5025420189,0.5263082385,0.7546547109,0.7696483731,0.0591981879,0.5949087143,0.8230769038,0.6409001350,0.7466957569,0.7896363139,0.8065621257,-0.6829438210,-0.6881237030,0.0291667009,-0.7407627702,-0.5948227644,-0.7083897591,-0.6993479133,-0.6773586273,-0.6413172483,0.4730041677,0.4749484062,0.0491644931,

In [4]:
NON_FEATURE_COLUMNS = {
    parcel_id_column, class_column, "parcel_id", "parcel_code", "class", "label", "label_code", "initial_label", "initial_label_code", "period_start", "period_end",
    "predicted_label", "predicted_label_code", "rfv_true", "svm_novelty_full", "parcel_area_m2", "eligible_pixel_count",
    "parcel_id", "period_start", "period_end", "geometry", "batch_number", "prediction_probability",
    "categ_code", "data_clean_mid", "has_rfv", "normal", "svm_outlier_full", "holdout_mid", "holdout_full", "meets_minimum_pixel_count"
}

BASE_COLUMNS = [parcel_id_column, class_column, "period_start", "geometry"]
remove_keywords = ["B02_", "B03_", "B04_", "count", "_mean", "_p25", "_p75", "_p10", "_p90", "_variance", "_coefficient_of_variation", "_bowley_skewness"] + list(NON_FEATURE_COLUMNS)

keep_statistics_columns = [x for x in list(gdf.columns) if all(keyword not in x for keyword in remove_keywords) and x not in BASE_COLUMNS]
keep_columns = BASE_COLUMNS + keep_statistics_columns
df_final = gdf[keep_columns].copy()
print(f"Columns dropped from the merged dataframe: {[x for x in list(gdf.columns) if x not in keep_columns]}")

numeric_or_boolean = set(df_final.select_dtypes(include=["number", "bool"]).columns)
features = [col for col in df_final.columns if col in numeric_or_boolean and col not in BASE_COLUMNS]

print(f"Identified {len(features)} features for modeling: {features}")

Columns dropped from the merged dataframe: ['period_end', 'eligible_pixel_count', 'meets_minimum_pixel_count', 'B02_mean', 'B02_median', 'B02_sd', 'B02_min', 'B02_max', 'B02_p10', 'B02_p25', 'B02_p75', 'B02_p90', 'B03_mean', 'B03_median', 'B03_sd', 'B03_min', 'B03_max', 'B03_p10', 'B03_p25', 'B03_p75', 'B03_p90', 'B04_mean', 'B04_median', 'B04_sd', 'B04_min', 'B04_max', 'B04_p10', 'B04_p25', 'B04_p75', 'B04_p90', 'B05_mean', 'B05_p10', 'B05_p25', 'B05_p75', 'B05_p90', 'B08_mean', 'B08_p10', 'B08_p25', 'B08_p75', 'B08_p90', 'B11_mean', 'B11_p10', 'B11_p25', 'B11_p75', 'B11_p90', 'B12_mean', 'B12_p10', 'B12_p25', 'B12_p75', 'B12_p90', 'EVI_mean', 'EVI_p10', 'EVI_p25', 'EVI_p75', 'EVI_p90', 'NDMI_mean', 'NDMI_p10', 'NDMI_p25', 'NDMI_p75', 'NDMI_p90', 'NDRE_mean', 'NDRE_p10', 'NDRE_p25', 'NDRE_p75', 'NDRE_p90', 'NDVI_mean', 'NDVI_p10', 'NDVI_p25', 'NDVI_p75', 'NDVI_p90', 'NDWI_mean', 'NDWI_p10', 'NDWI_p25', 'NDWI_p75', 'NDWI_p90', 'SAVI_mean', 'SAVI_p10', 'SAVI_p25', 'SAVI_p75', 'SAVI_p90'

In [5]:
label_counts = df_final[class_column].value_counts()
rare_labels = label_counts[label_counts < 100].index

df_final.loc[df_final[class_column].isin(rare_labels), class_column] = 9999
print(df_final[class_column].value_counts())

label
110     279768
119     114300
111      63588
129      43920
112      39456
130      35880
108      19812
128      16464
114      12012
116      10440
121       8100
101       7080
105       6384
122       5400
102       4380
106       3000
104       1836
117       1164
126        912
115        756
120        240
127        156
123        156
125        144
9999       120
Name: count, dtype: int64


In [6]:
duplicates = df_final.duplicated(subset=[parcel_id_column, "period_start"], keep=False)
print(f"Found {duplicates.sum()} duplicate rows based on parcel_id and period_start.")

Found 0 duplicate rows based on parcel_id and period_start.


In [7]:
df_final = df_final[df_final[class_column] != 9999].copy()

In [8]:
config = ClassificationPipelineConfig(
    project_dir=project_dir,
    target_column=class_column,
    feature_columns=features,
    id_column=parcel_id_column,
    
    time_column="period_start",
    reshape_time_series=True,
    prediction_cutoff="2024-09-30",
    
    apply_iqr=False,
    spatial_interpolation_method="nearest",
    spatial_interpolation_max_distance_in_meters=500,
        
    cv_folds=3,
    test_size=0.2,    
    spatial_split=True,
    spatial_split_grid_size=10,
    spatial_split_method="by_row",  # or "by_group"
    
    random_state=random_seed,
    n_jobs=1,
    max_search_candidates=12,
    interpretability_top_models=3,
    interpretability_include_shap=False,    
    top_voting_models=None,
    
    scoring_primary="f1_macro",
    selection_type="soft_voting",
    prediction_confidence_threshold=0.60,
        
    label_balancing_method="random_oversample",  # "smote",
    selected_models=(
        #"gradient_boosting",
        #"hist_gradient_boosting",
        "bagging",
        "random_forest",
        # "decision_tree",
        "extra_trees",
        # "bayesian",
        "xgboost",
        #"lightgbm",
        #"knn",
        "neural_network"
   ),
    max_map_geometries=20000,
    open_html_report=True,
)
pipeline = GeospatialClassificationPipeline(config)

In [9]:
summary_check = pipeline.run_check(df=df_final)
summary_check


Function <run_check> started on 2026-08-13 07:38:48

Checking input data...

Resetting project output folder: C:\work_dir\kozani_parcels\ml_classification
Deleting folder 'C:\work_dir\kozani_parcels\ml_classification\01_check'...
Successfully deleted folder 'C:\work_dir\kozani_parcels\ml_classification\01_check' on attempt 1
Deleting folder 'C:\work_dir\kozani_parcels\ml_classification\02_prepare'...
Successfully deleted folder 'C:\work_dir\kozani_parcels\ml_classification\02_prepare' on attempt 1
Deleting folder 'C:\work_dir\kozani_parcels\ml_classification\03_train'...
Successfully deleted folder 'C:\work_dir\kozani_parcels\ml_classification\03_train' on attempt 1
Deleting folder 'C:\work_dir\kozani_parcels\ml_classification\04_evaluate'...
Successfully deleted folder 'C:\work_dir\kozani_parcels\ml_classification\04_evaluate' on attempt 1
Deleting folder 'C:\work_dir\kozani_parcels\ml_classification\05_predict'...
Successfully deleted folder 'C:\work_dir\kozani_parcels\ml_classifica

{'rows_total': 56279,
 'rows_labeled': 56279,
 'rows_unknown_target': 0,
 'n_features_requested': 720,
 'n_features_active': 720,
 'numeric_features': ['B05_iqr__20231001',
  'B05_iqr__20231101',
  'B05_iqr__20231201',
  'B05_iqr__20240101',
  'B05_iqr__20240201',
  'B05_iqr__20240301',
  'B05_iqr__20240401',
  'B05_iqr__20240501',
  'B05_iqr__20240601',
  'B05_iqr__20240701',
  'B05_iqr__20240801',
  'B05_iqr__20240901',
  'B05_max__20231001',
  'B05_max__20231101',
  'B05_max__20231201',
  'B05_max__20240101',
  'B05_max__20240201',
  'B05_max__20240301',
  'B05_max__20240401',
  'B05_max__20240501',
  'B05_max__20240601',
  'B05_max__20240701',
  'B05_max__20240801',
  'B05_max__20240901',
  'B05_median__20231001',
  'B05_median__20231101',
  'B05_median__20231201',
  'B05_median__20240101',
  'B05_median__20240201',
  'B05_median__20240301',
  'B05_median__20240401',
  'B05_median__20240501',
  'B05_median__20240601',
  'B05_median__20240701',
  'B05_median__20240801',
  'B05_media

In [10]:
summary_prepare = pipeline.run_prepare()


Function <run_prepare> started on 2026-08-13 07:43:16

Preparing data...


E:\Eleni\git_repo\satellites\src\ml_classification\ml_classification_pipeline\visuals\maps.py:190: UserWarning: Legend does not support handles for PatchCollection instances.
See: https://matplotlib.org/stable/tutorials/intermediate/legend_guide.html#implementing-a-custom-legend-handler
  ax.legend()



Prepared train/test split: 45023 train rows, 11256 test rows.

Execution time of function <run_prepare> ended on 2026-08-13 07:43:56 - duration: 00:00:39


In [11]:
summary_train = pipeline.run_train()
summary_train


Function <run_train> started on 2026-08-13 07:43:56

Training models...
____________________________________________________________________________________________________

Training model random_forest


E:\Eleni\git_repo\satellites\src\ml_classification\ml_classification_pipeline\core\models.py:233: UserWarning: label_balancing_method was requested but imbalanced-learn is not installed. Falling back to no balancing.
  builder=lambda: _build_pipeline_with_optional_balancer(



Saved artifacts for random_forest: model=C:\work_dir\kozani_parcels\ml_classification\03_train\models\random_forest\best_model.joblib, cv_results=C:\work_dir\kozani_parcels\ml_classification\03_train\models\random_forest\cv_results.csv, plot=C:\work_dir\kozani_parcels\ml_classification\03_train\models\random_forest\search_results.png

Model random_forest finished in duration: 01:57:56
____________________________________________________________________________________________________

Training model extra_trees


E:\Eleni\git_repo\satellites\src\ml_classification\ml_classification_pipeline\core\models.py:252: UserWarning: label_balancing_method was requested but imbalanced-learn is not installed. Falling back to no balancing.
  builder=lambda: _build_pipeline_with_optional_balancer(



Saved artifacts for extra_trees: model=C:\work_dir\kozani_parcels\ml_classification\03_train\models\extra_trees\best_model.joblib, cv_results=C:\work_dir\kozani_parcels\ml_classification\03_train\models\extra_trees\cv_results.csv, plot=C:\work_dir\kozani_parcels\ml_classification\03_train\models\extra_trees\search_results.png

Model extra_trees finished in duration: 00:02:11
____________________________________________________________________________________________________

Training model neural_network


...

E:\Eleni\git_repo\satellites\src\ml_classification\ml_classification_pipeline\core\models.py:341: UserWarning: label_balancing_method was requested but imbalanced-learn is not installed. Falling back to no balancing.
  builder=lambda: _build_pipeline_with_optional_balancer(



Saved artifacts for bagging: model=C:\work_dir\kozani_parcels\ml_classification\03_train\models\bagging\best_model.joblib, cv_results=C:\work_dir\kozani_parcels\ml_classification\03_train\models\bagging\cv_results.csv, plot=C:\work_dir\kozani_parcels\ml_classification\03_train\models\bagging\search_results.png

Model bagging finished in duration: 00:54:13
____________________________________________________________________________________________________

Training model xgboost


E:\Eleni\git_repo\satellites\src\ml_classification\ml_classification_pipeline\core\models.py:398: UserWarning: label_balancing_method was requested but imbalanced-learn is not installed. Falling back to no balancing.
  builder=lambda: _build_pipeline_with_optional_balancer(



Saved artifacts for xgboost: model=C:\work_dir\kozani_parcels\ml_classification\03_train\models\xgboost\best_model.joblib, cv_results=C:\work_dir\kozani_parcels\ml_classification\03_train\models\xgboost\cv_results.csv, plot=C:\work_dir\kozani_parcels\ml_classification\03_train\models\xgboost\search_results.png

Model xgboost finished in duration: 00:04:58

Best CV model: random_forest using score_minus_std=0.1698

Execution time of function <run_train> ended on 2026-08-13 10:48:36 - duration: 03:04:40


{'models_trained': ['random_forest',
  'extra_trees',
  'bagging',
  'neural_network',
  'xgboost'],
 'models_failed': [],
 'best_cv_model': 'random_forest',
 'cv_ranking_method': 'score_minus_std'}

In [12]:
summary_evaluate = pipeline.run_evaluate()
summary_evaluate


Function <run_evaluate> started on 2026-08-13 10:48:36

Evaluating models...


c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  y_pred,
c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  y_pred,
c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  y_pred,
c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0


Selected strategy: soft_voting using ['random_forest', 'extra_trees', 'bagging', 'neural_network', 'xgboost']

Execution time of function <run_evaluate> ended on 2026-08-13 10:49:36 - duration: 00:00:59


{'selection_type': 'soft_voting',
 'selected_models': ['random_forest',
  'extra_trees',
  'bagging',
  'neural_network',
  'xgboost'],
 'selected_metric': 'cv_score_minus_std',
 'selected_score': 0.1280397126570926,
 'selection_source': 'cross_validation',
 'labels': ['101',
  '102',
  '104',
  '105',
  '106',
  '108',
  '110',
  '111',
  '112',
  '114',
  '115',
  '116',
  '117',
  '119',
  '120',
  '121',
  '122',
  '123',
  '125',
  '126',
  '127',
  '128',
  '129',
  '130'],
 '_schema': {'artifact': 'selection_summary', 'schema_version': '1.0.0'}}

In [13]:
summary_predict = pipeline.run_predict()
summary_predict


Function <run_predict> started on 2026-08-13 10:49:36

Classify all data...


E:\Eleni\git_repo\satellites\src\ml_classification\ml_classification_pipeline\core\models.py:233: UserWarning: label_balancing_method was requested but imbalanced-learn is not installed. Falling back to no balancing.
  builder=lambda: _build_pipeline_with_optional_balancer(
E:\Eleni\git_repo\satellites\src\ml_classification\ml_classification_pipeline\core\models.py:252: UserWarning: label_balancing_method was requested but imbalanced-learn is not installed. Falling back to no balancing.
  builder=lambda: _build_pipeline_with_optional_balancer(
E:\Eleni\git_repo\satellites\src\ml_classification\ml_classification_pipeline\core\models.py:341: UserWarning: label_balancing_method was requested but imbalanced-learn is not installed. Falling back to no balancing.
  builder=lambda: _build_pipeline_with_optional_balancer(
E:\Eleni\git_repo\satellites\src\ml_classification\ml_classification_pipeline\core\models.py:322: UserWarning: label_balancing_method was requested but imbalanced-learn is not


Filled 0 unknown labels.

Execution time of function <run_predict> ended on 2026-08-13 14:25:16 - duration: 03:35:40


{'selection_type': 'soft_voting',
 'selected_models': ['random_forest',
  'extra_trees',
  'bagging',
  'neural_network',
  'xgboost'],
 'rows_filled': 0,
 'rows_unknown_original': 0,
 'rows_needing_review': 18192,
 'prediction_confidence_threshold': 0.6,
 'output_path': 'C:\\work_dir\\kozani_parcels\\ml_classification\\05_predict\\final_predictions.joblib',
 '_schema': {'artifact': 'predict_summary', 'schema_version': '1.0.0'}}

In [31]:
# Show only rows that were unknown during training
classified_gdf = pd.read_csv(os.path.join(project_dir, "05_predict", "final_predictions_preview.csv"))
classified_gdf[parcel_id_column] = classified_gdf[parcel_id_column].astype(str)
print(len(classified_gdf), "rows in classified_gdf")
display(classified_gdf.head())

56279 rows in classified_gdf


,parcel_code,label,label_prediction,prediction_confidence,prediction_needs_review,label_filled
0,100010810,110,110,0.4003046146,True,110
1,100010811,110,110,0.5746935511,True,110
2,100010812,129,129,0.5588624383,True,129
3,100010816,119,119,0.8170398110,False,119
4,100010817,129,129,0.6301571802,False,129


In [33]:
from ml_classification.ml_classification_pipeline.core.persistence import load_joblib
test_df = load_joblib(Path(f"{project_dir}/02_prepare/test_dataset.joblib"))
test_df[parcel_id_column] = test_df[parcel_id_column].astype(str)
print(len(test_df), "rows in test_df")
test_df.head()

11256 rows in test_df


,parcel_code,B05_iqr__20231001,B05_iqr__20231101,B05_iqr__20231201,B05_iqr__20240101,B05_iqr__20240201,B05_iqr__20240301,B05_iqr__20240401,B05_iqr__20240501,B05_iqr__20240601,B05_iqr__20240701,B05_iqr__20240801,B05_iqr__20240901,B05_max__20231001,B05_max__20231101,B05_max__20231201,B05_max__20240101,B05_max__20240201,B05_max__20240301,B05_max__20240401,B05_max__20240501,B05_max__20240601,B05_max__20240701,B05_max__20240801,B05_max__20240901,B05_median__20231001,B05_median__20231101,B05_median__20231201,B05_median__20240101,B05_median__20240201,B05_median__20240301,B05_median__20240401,B05_median__20240501,B05_median__20240601,B05_median__20240701,B05_median__20240801,B05_median__20240901,B05_min__20231001,B05_min__20231101,B05_min__20231201,B05_min__20240101,B05_min__20240201,B05_min__20240301,B05_min__20240401,B05_min__20240501,B05_min__20240601,B05_min__20240701,B05_min__20240801,B05_min__20240901,B05_range__20231001,B05_range__20231101,B05_range__20231201,B05_range__20240101,B05_range__20240201,B05_range__20240301,B05_range__20240401,B05_range__20240501,B05_range__20240601,B05_range__20240701,B05_range__20240801,B05_range__20240901,B05_sd__20231001,B05_sd__20231101,B05_sd__20231201,B05_sd__20240101,B05_sd__20240201,B05_sd__20240301,B05_sd__20240401,B05_sd__20240501,B05_sd__20240601,B05_sd__20240701,B05_sd__20240801,B05_sd__20240901,B08_iqr__20231001,B08_iqr__20231101,B08_iqr__20231201,B08_iqr__20240101,B08_iqr__20240201,B08_iqr__20240301,B08_iqr__20240401,B08_iqr__20240501,B08_iqr__20240601,B08_iqr__20240701,B08_iqr__20240801,B08_iqr__20240901,B08_max__20231001,B08_max__20231101,B08_max__20231201,B08_max__20240101,B08_max__20240201,B08_max__20240301,B08_max__20240401,B08_max__20240501,B08_max__20240601,B08_max__20240701,B08_max__20240801,B08_max__20240901,B08_median__20231001,B08_median__20231101,B08_median__20231201,B08_median__20240101,B08_median__20240201,B08_median__20240301,B08_median__20240401,B08_median__20240501,B08_median__20240601,B08_median__20240701,B08_median__20240801,B08_median__20240901,B08_min__20231001,B08_min__20231101,B08_min__20231201,B08_min__20240101,B08_min__20240201,B08_min__20240301,B08_min__20240401,B08_min__20240501,B08_min__20240601,B08_min__20240701,B08_min__20240801,B08_min__20240901,B08_range__20231001,B08_range__20231101,B08_range__20231201,B08_range__20240101,B08_range__20240201,B08_range__20240301,B08_range__20240401,B08_range__20240501,B08_range__20240601,B08_range__20240701,B08_range__20240801,B08_range__20240901,B08_sd__20231001,B08_sd__20231101,B08_sd__20231201,B08_sd__20240101,B08_sd__20240201,B08_sd__20240301,B08_sd__20240401,B08_sd__20240501,B08_sd__20240601,B08_sd__20240701,B08_sd__20240801,B08_sd__20240901,B11_iqr__20231001,B11_iqr__20231101,B11_iqr__20231201,B11_iqr__20240101,B11_iqr__20240201,B11_iqr__20240301,B11_iqr__20240401,B11_iqr__20240501,B11_iqr__20240601,B11_iqr__20240701,B11_iqr__20240801,B11_iqr__20240901,B11_max__20231001,B11_max__20231101,B11_max__20231201,B11_max__20240101,B11_max__20240201,B11_max__20240301,B11_max__20240401,B11_max__20240501,B11_max__20240601,B11_max__20240701,B11_max__20240801,B11_max__20240901,B11_median__20231001,B11_median__20231101,B11_median__20231201,B11_median__20240101,B11_median__20240201,B11_median__20240301,B11_median__20240401,B11_median__20240501,B11_median__20240601,B11_median__20240701,B11_median__20240801,B11_median__20240901,B11_min__20231001,B11_min__20231101,B11_min__20231201,B11_min__20240101,B11_min__20240201,B11_min__20240301,B11_min__20240401,B11_min__20240501,B11_min__20240601,B11_min__20240701,B11_min__20240801,B11_min__20240901,B11_range__20231001,B11_range__20231101,B11_range__20231201,B11_range__20240101,B11_range__20240201,B11_range__20240301,B11_range__20240401,B11_range__20240501,B11_range__20240601,B11_range__20240701,B11_range__20240801,B11_range__20240901,B11_sd__20231001,B11_sd__20231101,B11_sd__20231201,B11_sd__20240101,B11_sd__20240201,B11_sd__20240301,B11_sd__20240401,B11_sd__20240501,B11_sd__20240601,B11_sd

In [34]:
test_ids =list(test_df[parcel_id_column].unique())
classified_gdf = classified_gdf[classified_gdf[parcel_id_column].isin(test_ids)]
print(len(classified_gdf), "rows in classified_gdf after filtering with test_df")
classified_gdf[f"accuracy"] = classified_gdf[class_column] == classified_gdf[f"{class_column}_prediction"]
accuracy = classified_gdf[f"accuracy"].mean()
print(f"Accuracy on data: {accuracy:.2%}")

11256 rows in classified_gdf after filtering with test_df
Accuracy on data: 98.45%
